# Capstone — A GAN that generates digits (MNIST)

_Capstones_

**Walk five papers, building piece by piece a conditional DCGAN that draws whatever MNIST digit you ask for.**

---

This notebook is a practice scaffold for the **Capstone — A GAN that generates digits (MNIST)** lesson. We assemble the conditional DCGAN one piece at a time — the generator, the discriminator, the data pipeline, and the adversarial training loop — then watch a requested digit emerge from noise. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the MNIST dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.MNIST(root="./data", train=True, download=True)
print("dataset: MNIST   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — Set the hyperparameters and check the GAN math

We fix the seed and the dimensions: the noise size `NZ`, the generator/discriminator channel widths, and the label-embedding size `EMB`. Before building the networks we re-derive two facts. First, at the GAN optimum the generator's distribution matches the data, the discriminator can only guess `D = 1/2`, and its BCE loss parks at `2·log2 = -log4`; with conditioning this holds per requested class. Second, a transposed conv with kernel 4, stride 2, padding 1 **doubles** the spatial size — the building block of the up-sampler. The `assert` confirms the formula against a real layer.

In [ ]:
import torch
import torch.nn as nn
import math
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
K  = 10                                   # MNIST has 10 digit classes (0..9)
NZ, NGF, NDF, NC = 100, 64, 64, 1         # noise dim, generator/discriminator widths, image channels
EMB = 50                                  # size of the learned label embedding

# --- Recompute the worked-example numbers (they match the CODEVIZ caption). ---
# At the GAN optimum p_g = p_data the discriminator is reduced to guessing, D = 1/2, and the
# discriminator's binary-cross-entropy loss parks at 2*log2 = -log4. With conditioning this holds
# PER REQUESTED CLASS y. A transposed conv with k=4,s=2,p=1 DOUBLES the spatial size.
def convT_out(h, k, s, p):
    return (h - 1) * s - 2 * p + k        # H_out = (H_in-1)s - 2p + k

equilibrium_loss = 2 * math.log(2)        # discriminator BCE at the optimum
print("equilibrium D-loss 2*log2 = %.4f   (= -log4 = %.4f)" % (equilibrium_loss, -math.log(4)))
doubled = convT_out(4, 4, 2, 1)           # 4x4 input -> 8x8 output
print("convT(4x4, k4 s2 p1) -> %dx%d (doubles)" % (doubled, doubled))
probe = nn.ConvTranspose2d(8, 8, 4, 2, 1)
assert convT_out(4, 4, 2, 1) == 8 == probe(torch.zeros(1, 8, 4, 4)).shape[-1]

### Step 2 — Build the conditional generator and discriminator

Both networks are conditioned on the digit label (the cGAN idea). The **generator** embeds the label into a `1x1` plane, concatenates it onto the noise, and up-samples through transposed convs (BatchNorm + ReLU, Tanh output) from `1x1` to `32x32` — a fully-convolutional DCGAN backbone with no dense layer. The **discriminator** embeds the label into a full `32x32` plane, concatenates it as an extra image channel, and down-samples to a single real/fake logit. `init_weights` applies the DCGAN initialization: zero-centered normal with std 0.02.

In [ ]:
# --- 1. CONDITIONAL DCGAN GENERATOR: [noise | label-plane] -> 32x32 image. ---
# Label conditioning (cGAN, step 5): embed y, reshape to a 1x1 "plane", concat onto the noise.
# Backbone (DCGAN, step 2): all transposed-conv up-sampler, BatchNorm (step 3) + ReLU, Tanh out.
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_emb = nn.Embedding(K, EMB)                 # learned vector per digit class
        self.net = nn.Sequential(
            nn.ConvTranspose2d(NZ + EMB, NGF*4, 4, 1, 0, bias=False),  # 1x1 -> 4x4 (no FC layer)
            nn.BatchNorm2d(NGF*4), nn.ReLU(True),
            nn.ConvTranspose2d(NGF*4, NGF*2, 4, 2, 1, bias=False),     # 4 -> 8
            nn.BatchNorm2d(NGF*2), nn.ReLU(True),
            nn.ConvTranspose2d(NGF*2, NGF,   4, 2, 1, bias=False),     # 8 -> 16
            nn.BatchNorm2d(NGF),   nn.ReLU(True),
            nn.ConvTranspose2d(NGF, NC,      4, 2, 1, bias=False),     # 16 -> 32
            nn.Tanh())                                                 # output: NO BatchNorm, Tanh -> [-1,1]
    def forward(self, z, y):                                  # z:(N,NZ,1,1)  y:(N,)
        e = self.label_emb(y).view(z.size(0), EMB, 1, 1)      # label -> (N,EMB,1,1) plane
        return self.net(torch.cat([z, e], dim=1))             # concat label onto noise channels

# --- 2. CONDITIONAL DCGAN DISCRIMINATOR: [image | label-plane] -> one real/fake logit. ---
# Same conditioning idea: embed y, broadcast to a 32x32 plane, concat as an extra image channel.
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_emb = nn.Embedding(K, 32*32)               # one value per pixel, per class
        self.net = nn.Sequential(
            nn.Conv2d(NC + 1, NDF, 4, 2, 1, bias=False),      # 32 -> 16  (input layer: NO BatchNorm)
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(NDF, NDF*2, 4, 2, 1, bias=False),       # 16 -> 8
            nn.BatchNorm2d(NDF*2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(NDF*2, NDF*4, 4, 2, 1, bias=False),     # 8 -> 4
            nn.BatchNorm2d(NDF*4), nn.LeakyReLU(0.2, True),
            nn.Conv2d(NDF*4, 1, 4, 1, 0, bias=False))         # 4 -> 1x1 logit
    def forward(self, x, y):                                  # x:(N,1,32,32)  y:(N,)
        e = self.label_emb(y).view(x.size(0), 1, 32, 32)      # label -> (N,1,32,32) plane
        return self.net(torch.cat([x, e], dim=1)).view(-1)    # concat label as an extra channel

# DCGAN weight init: zero-centered normal, std 0.02 (paper-dcgan, Section 4).
def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

G = Generator().to(device)
G.apply(init_weights)
D = Discriminator().to(device)
D.apply(init_weights)

### Step 3 — Prepare the data and a way to *see* conditioning

MNIST is resized 28→32 and normalized to `[-1, 1]` so it matches the generator's Tanh output. We use BCE-with-logits and one Adam optimizer per network with the DCGAN betas `(0.5, 0.999)`. A single `fixed_z` noise vector is reused across epochs so we can watch the *same* seed sharpen over time. The `preview` helper asks the generator for a chosen digit class and prints it as ASCII art — letting us literally see whether the label conditioning is working.

In [ ]:
# --- 3. MNIST scaled to [-1,1] to match Tanh, padded 28 -> 32. ---
tfm = T.Compose([T.Resize(32), T.ToTensor(), T.Normalize((0.5,), (0.5,))])
data = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
loader = torch.utils.data.DataLoader(data, batch_size=128, shuffle=True, drop_last=True)

# --- 4. PRINT a generated digit of a CHOSEN class as ASCII, so we SEE conditioning work. ---
bce = nn.BCEWithLogitsLoss()
optD = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
optG = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
fixed_z = torch.randn(1, NZ, 1, 1, device=device)            # same noise -> compare across epochs

def preview(want):                                           # want = the digit class we REQUEST
    G.eval()
    with torch.no_grad():
        img = G(fixed_z, torch.tensor([want], device=device))[0, 0].cpu()
    G.train()
    img = (img + 1) / 2                                      # [-1,1] -> [0,1] for printing
    print("--- requested digit:", want, "---")
    for r in range(2, 32, 2):                                # subsample rows for a compact view
        print("".join(" .:-=+*#%@"[min(9, int(img[r, c].clamp(0,1) * 9))] for c in range(2, 32)))

### Step 4 — Run the adversarial loop and generate on demand

Each step trains the discriminator to call real images (under their true labels) "real" and generated images (under the labels they were generated for) "fake", then trains the generator with the **non-saturating** objective to make the discriminator call its class-`yf` fakes "real for yf". The label is threaded through both networks, so after training we can request any specific digit. Watch the discriminator loss settle near `2·log2 ≈ 1.386` per class as the requested digit emerges from noise.

In [ ]:
# --- 5. THE ADVERSARIAL LOOP (step 1), with the label threaded through BOTH nets (step 5). ---
def train(epochs=3):
    real_mean = next(iter(loader))[0].mean().item()
    print("real data pixel mean ~ %.3f  (target for generated samples)\n" % real_mean)
    step = 0
    for ep in range(epochs):
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            n = x.size(0)
            real_t = torch.ones(n, device=device)
            fake_t = torch.zeros(n, device=device)
            # (a) DISCRIMINATOR step: reals under true labels -> 1; fakes under same labels -> 0.
            z  = torch.randn(n, NZ, 1, 1, device=device)
            yf = torch.randint(0, K, (n,), device=device)     # labels we generate fakes for
            fake = G(z, yf)
            lossD = bce(D(x, y), real_t) + bce(D(fake.detach(), yf), fake_t)  # detach: don't move G here
            optD.zero_grad()
            lossD.backward()
            optD.step()
            # (b) GENERATOR step: NON-SATURATING -- make D call class-yf fakes "real for yf".
            lossG = bce(D(fake, yf), real_t)
            optG.zero_grad()
            lossG.backward()
            optG.step()
            if step % 200 == 0:
                with torch.no_grad():
                    s = G(fixed_z, torch.tensor([ep % K], device=device))
                print("step %5d  D %.3f  G %.3f  sample(mean %+.3f std %.3f)"
                      % (step, lossD.item(), lossG.item(), s.mean().item(), s.std().item()))
            step += 1
        preview(want=ep % K)                                  # ask for a different digit each epoch

print("\nbefore training (requesting a 7) -- pure noise:")
preview(7)
train(epochs=3)
print("\nNow generate any digit ON DEMAND:")
for d in [0, 5, 9]:
    preview(d)
print("\nD loss settles near 2*log2 = 1.386 PER CLASS; the requested digit emerges from noise.")
# Numbers vary by hardware/seed -- this is OUR small run, not the paper's reported result.

# --- 6. ABLATION: drop the label from BOTH nets -> a plain DCGAN you can no longer steer. ---
# Rebuild G/D without the embedding/concat (the cGAN mechanism) and train with the same loop.
# Generation still works, but requesting a class is impossible -- G's only input is noise, so the
# digit is whatever z maps to. Removing the conditioning removes the on-demand-class knob.